# Search in ADS
We begin our fist part by searching an exporting the desired publications. Apart form the above linked basis for this part, other important References to other ADS API Jupyter notebooks, are especially the ones on [searching](https://github.com/adsabs/adsabs-dev-api/blob/master/API_documentation_Python/Search_API_Python.ipynb), [exporting](https://github.com/adsabs/adsabs-dev-api/blob/master/API_documentation_Python/Export_API_Python.ipynb), and [using the API with Python](https://github.com/adsabs/adsabs-dev-api/blob/master/API_documentation_UNIXshell/Converting_curl_to_python.ipynb).

## Imports
We start by importing the required Python Packages for this Notebook and by loading our personal [API token](https://ui.adsabs.harvard.edu/user/settings/token), which has to be created beforhand.

Let's test a reference @Danielmeyer1991 [p. 33], and a footnote[^longnote].

[^longnote]: Here's one with multiple blocks.  
    Subsequent paragraphs are indented to show that they belong to the previous footnote.

In [1]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime
import os
import pickle
import time
import shutil

# Load the environment variables from the .env file
load_dotenv()

# Access to the environment variable
token = os.getenv('ADS_API_KEY')

# --- Global Configuration ---
DATA_DIR = Path("2_data")
RUN_PREFIX = "bibcodes_references_"
RUN_GLOB = f"{RUN_PREFIX}*.pkl"
LATEST_FILE = DATA_DIR / f"{RUN_PREFIX}latest.pkl"

# Set to True if you want to force a new API fetch and create a new timestamped run.
# Set to False to load the latest existing run (if available).
FORCE_REFRESH = True

## Curating the Library and defining the Search
Our Search is based on a core Library we curated in the frontend of ADS. It contains papers by Einstein (curated) that touch topics related to general relativity, cosmology and unified field theory (`Set_A`). We then define a `Set_B` which queries all papers that cite at least one of the papers in `Set_A` (via the ADS `citations()` operator) between the years 1915 and 2000. We additionally define a 2-hop expansion `Set_C` as `citations(citations(Set_A))`.

Example (1-hop): `citations(docs(library/P0hyiLw0T8qsyHSBpWigJA)) AND year:1915-2000`

Our final `query` variable will be a combination of six different search variables, defined below.

In [2]:
# Treder's papers on GR. Curated by hand and saved as a library in ADS.
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

In [3]:
# Einstein's papers on GR. Curated by hand and saved as a library in ADS.
Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"

# Citation expansion: use citations(query) to get *all* citing papers.
# (The second-order operator reviews(query) is ranked-by-frequency and is not the same thing.)

# 1-hop: papers citing the Set_A seed set
Set_B = f"citations({Set_A}) AND year:1915-2000"

# 2-hop: papers citing papers that cite Set_A
# Kept as citations(citations(Set_A)) so it stays syntactically simple (no nested year filters).
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"

# Combined set of papers from Set_A, Set_B, and Set_C
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"

# abs: searches abstract + title + keywords
# Keep this as a single-line query (no newlines/indentation) to avoid parser edge cases.
gr_terms = [
    '(general AND relativi*)',
    'gravit*',
    '(allgemein* AND relativi*)',
    '\"relativité générale\"',
    '\"teoria della relatività\"',
    '\"Gravité quantique\"',
    '\"Gravità quantistica\"',
    '(einheitlich* AND feld*)',
    'Quantengravitation',
    '\"champ unifié\"',
    '(unified AND field*)',
    '\"quantum gravity\"',
    'cosmo*',
    'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gr_terms)}) AND year:1911-2000"

# Combined set of papers from Set_D, Set_T and Set_E
Set_F = f"({Set_D}) OR ({Set_T}) OR ({Set_E})"

To convert these to a search, we first have to take the base URL for searching in the API: `https://api.adsabs.harvard.edu/v1/search/query?`

We'll be exporting all of the search results plus references, so we return the bibcode + references of those bicodes for each search result. The search terms are then URL-encoded before we add them to the URL.

In [4]:
# Define the search query to be used.
query = Set_F

# The urlencode function can take a dictionary of key-value pairs and convert it to a URL-encoded string.
from urllib.parse import urlencode

# The dictionary contains two items:
#   - The key 'q' (for query) with the value of the search query defined above.
#   - The key 'fl' (for fields list) with the value 'bibcode,reference', indicating that these fields should be returned in the results.
encoded_query = urlencode({'q': query, 'fl': 'bibcode,reference,esources'})

The encoded search terms are then added to the base URL to create the URL we'll send our API request to. We return the results in JSON format.

In [5]:
# The URL is formed by appending the encoded_query to the base URL of the API.
# The headers dictionary is used to provide the authorization token needed to access the API.
results = requests.get(
    "https://api.adsabs.harvard.edu/v1/search/query?{}".format(encoded_query),
    headers={'Authorization': 'Bearer ' + token}
)

# The result is printed out, providing a readable representation of the response data.
results.json()

{'responseHeader': {'status': 0,
  'QTime': 2752,
  'params': {'q': '((docs(library/P0hyiLw0T8qsyHSBpWigJA)) OR (citations(docs(library/P0hyiLw0T8qsyHSBpWigJA)) AND year:1915-2000) OR (citations(citations(docs(library/P0hyiLw0T8qsyHSBpWigJA))) AND year:1915-2000)) OR ((docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)) OR (abs:((general AND relativi*) OR gravit* OR (allgemein* AND relativi*) OR "relativité générale" OR "teoria della relatività" OR "Gravité quantique" OR "Gravità quantistica" OR (einheitlich* AND feld*) OR Quantengravitation OR "champ unifié" OR (unified AND field*) OR "quantum gravity" OR cosmo* OR Kosmo*) AND year:1911-2000)',
   'fl': 'bibcode,reference,esources',
   'start': '0',
   'internal_logging_params': 'X-Amzn-Trace-Id=Root=1-696f6a0a-246540f02bbfc01c4fbb2243',
   'rows': '10',
   'wt': 'json'}},
 'response': {'numFound': 180381,
  'start': 0,
  'numFoundExact': True,
  'docs': [{'bibcode': '1998LRR.....1....1R',
    'reference': ['1957PhRv..108.1063R'

## Fetching the search results
A query returns 10 results by default (see the `rows` parameter in the response), but there is actually a total of around 109000 results that match the search (see the `numFound` parameter in the response). The ADS API gives a couple of options to return all of the search results, but paging through the list of results is the only viable option for numbers this large. We'll also extract the bibcodes from the references out of the dictionary structure they were returned in into a list. 

To do this we define a function `fetch_bibcodes_references_esources` that takes the `encoded_query` as an argument, pages through the list of results and returns a list of all bibcodes, references and esources for that query. For speed and stability it uses a persistent HTTP session, retries with backoff on transient errors / rate limiting, and (by default) `cursorMark`-based paging for deep result sets.

In [ ]:
import time
import requests
import pandas as pd
from urllib.parse import parse_qsl, urlencode

def fetch_bibcodes_references_esources(encoded_query, token, rows=2000):
    """
    Fetches bibcodes, references, and esources using cursorMark for deep paging.
    Includes persistent session and retries with backoff.
    """
    session = requests.Session()
    session.headers.update({'Authorization': 'Bearer ' + token})
    
    # Base search URL
    url = "https://api.adsabs.harvard.edu/v1/search/query"
    
    # Extract existing params
    params = dict(parse_qsl(encoded_query))
    
    # Clean up params for cursor-based paging:
    # 1. Remove 'start' if present (cursorMark uses cursorMark instead)
    params.pop('start', None)
    # 2. Add rows and a unique sort (required for cursorMark)
    params['rows'] = rows
    params['sort'] = 'date desc, id asc'  # include uniqueKey tie-breaker
    
    bibcodes, references, esources, fulltext_urls = [], [], [], []
    cursor = "*"  # Start of deep paging
    
    print(f"Starting retrieval...")
    
    while True:
        params['cursorMark'] = cursor
        
        # Retry loop for transient errors (429, 5xx)
        for attempt in range(5):
            response = session.get(url, params=params)
            
            if response.status_code == 200:
                break
            
            # Detailed Error Reporting for 400 Bad Request
            if response.status_code == 400:
                print(f"\n--- API 400 BAD REQUEST ---")
                print(f"URL: {response.url}")
                try: 
                    err_json = response.json()
                    print(f"Error Detail: {err_json.get('error', 'No detail provided')}")
                except: 
                    print(f"Response text: {response.text[:500]}")
                raise requests.exceptions.HTTPError(response=response)
                
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 60))
                print(f"\nRate limited. Waiting {wait}s...")
                time.sleep(wait)
            elif response.status_code >= 500:
                wait = 2 ** attempt
                print(f"\nServer error {response.status_code}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                response.raise_for_status()
        else:
            print("\nFailed after multiple retries.")
            break

        data = response.json()
        docs = data.get('response', {}).get('docs', [])
        if not docs:
            break

        for doc in docs:
            bc = doc.get('bibcode')
            bibcodes.append(bc)
            references.append(doc.get('reference', []))
            esources.append(doc.get('esources', []))
            
            # Identify fulltext PDF links
            pdf = next((f"https://ui.adsabs.harvard.edu/link_gateway/{bc}/{res}"
                        for res in doc.get('esources', []) 
                        if any(p in res for p in ['ADS_PDF', 'PUB_PDF'])), None)
            fulltext_urls.append(pdf)

        next_cursor = data.get('nextCursorMark')
        if next_cursor == cursor or not next_cursor:
            break
        cursor = next_cursor
        print(f"Progress: {len(bibcodes)} records fetched...", end="\r")
    
    print(f"\nDone. Total retrieved: {len(bibcodes)}")
    return bibcodes, references, esources, fulltext_urls

We define a function to get fulltext URLs for a given bibcode, preferring ADS_PDF and PUB_PDF over other formats. We use threading to speed up this process.

We start by fetching the bibcodes, references, and esources using the function defined earlier. This process might take a few minutes due to the large amount of data.

In [7]:
%%time

def _list_runs():
    runs = sorted(DATA_DIR.glob(RUN_GLOB), key=lambda p: p.stat().st_mtime, reverse=True)
    # Exclude the 'latest' alias from timestamp sorting
    runs = [p for p in runs if p.name != LATEST_FILE.name]
    return runs

def _latest_run_path():
    if LATEST_FILE.exists():
        return LATEST_FILE
    runs = _list_runs()
    return runs[0] if runs else None

# Check if we should load from cache or fetch new data
if not FORCE_REFRESH:
    latest = _latest_run_path()
    if latest is None or not latest.exists():
        raise FileNotFoundError("No existing runs found in 2_data. Set FORCE_REFRESH=True to create one.")
    print(f"Loading latest run from: {latest}")
    with open(latest, "rb") as f:
        bibcodes, references, esources, fulltext_urls = pickle.load(f)
    print(f"Loaded {len(bibcodes)} records.")
else:
    # Create a new timestamped run
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_path = DATA_DIR / f"{RUN_PREFIX}{ts}.pkl"
    
    try:
        bibcodes, references, esources, fulltext_urls = fetch_bibcodes_references_esources(encoded_query, token)
        if bibcodes:
            with open(run_path, "wb") as f:
                pickle.dump((bibcodes, references, esources, fulltext_urls), f)
            # Update latest alias (copy, not move)
            shutil.copy(run_path, LATEST_FILE)
            print(f"New run saved: {run_path}")
            print(f"Latest updated: {LATEST_FILE}")
        else:
            print("Fetch returned no results. No new run created.")
    except Exception as e:
        print(f"CRITICAL ERROR during fetch: {e}")
        latest = _latest_run_path()
        if latest and latest.exists():
            print("Preserving existing data. Loading latest run...")
            with open(latest, "rb") as f:
                bibcodes, references, esources, fulltext_urls = pickle.load(f)
        else:
            raise

Starting retrieval...
Progress: 180381 records fetched...
Done. Total retrieved: 180381
New run saved: 2_data\bibcodes_references_20260120_124208.pkl
Latest updated: 2_data\bibcodes_references_latest.pkl
CPU times: total: 3.83 s
Wall time: 5min 33s


We then verify the quantity of exported bibcodes and confirm that the length of the bibcodes list matches that of the references list to check that so far everything worked as expected.

In [8]:
# Verify the lengths of the lists
if len(bibcodes) == len(references) == len(esources):
    print(f'Lists of Bibcodes, References, and Esources have equal length of: {len(bibcodes)}')
else:
    print("Attention! Lists of Bibcodes, References, and Esources have different lengths")

print(type(bibcodes), type(bibcodes[100]), bibcodes[100])
print(type(references), type(references[100]), references[100])
print(type(esources), type(esources[100]), esources[100])

Lists of Bibcodes, References, and Esources have equal length of: 180381
<class 'list'> <class 'str'> 2000cond.mat.12316S
<class 'list'> <class 'list'> ['1996MPLB...10..999S', '1998JETPL..67..881V']
<class 'list'> <class 'list'> ['EPRINT_HTML', 'EPRINT_PDF']


We have saved the query results their references and esources into lists. Now we can extract the information we want by exporting the relevant fields from the ADS API. We'll be exporting ``'Bibcode', 'Author', 'Title', 'Year', 'Journal', 'Journal Abbreviation', 'Issue', 'Volume', 'First Page', 'Last Page', 'Abstract', 'Keywords', 'DOI', 'Affiliation', 'Category', 'Citation Count'`` for each paper. We'll also export the same information for each reference and will need to use a custom format to do this.

In [9]:
# --- TEMP: compare latest vs previous run (safe to delete after running) ---
from __future__ import annotations

import pickle
from pathlib import Path
from collections import Counter

def _list_runs():
    runs = sorted(DATA_DIR.glob(RUN_GLOB), key=lambda p: p.stat().st_mtime, reverse=True)
    # Exclude the 'latest' alias from timestamp sorting
    runs = [p for p in runs if p.name != LATEST_FILE.name]
    return runs

runs = _list_runs()
new_path = runs[0] if len(runs) >= 1 else None
old_path = runs[1] if len(runs) >= 2 else None

def _load_pickle(path: Path):
    with path.open("rb") as f:
        data = pickle.load(f)
    if not (isinstance(data, tuple) and len(data) >= 2):
        raise ValueError(f"Unexpected pickle format in {path}: {type(data)}")
    bibcodes = list(data[0])
    references = list(data[1])
    esources = list(data[2]) if len(data) > 2 else []
    fulltext_urls = list(data[3]) if len(data) > 3 else []
    return bibcodes, references, esources, fulltext_urls

def _as_set(x):
    return set(x) if x is not None else set()

def _flatten_refs(ref_lists):
    # ref_lists is list-of-lists (or missing/None); return a flat list
    out = []
    for item in ref_lists:
        if not item:
            continue
        # Some records might store references as tuple/np array; treat them as iterables
        try:
            out.extend(list(item))
        except TypeError:
            continue
    return out

def _jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

if new_path is None:
    print("No runs found in 2_data. Create one with FORCE_REFRESH=True.")
elif old_path is None:
    print(f"Only one run found: {new_path}")
    print("Create a second run (FORCE_REFRESH=True) to compare.")
else:
    new_bib, new_refs, new_esrc, new_ft = _load_pickle(new_path)
    old_bib, old_refs, old_esrc, old_ft = _load_pickle(old_path)

    new_bib_set = _as_set(new_bib)
    old_bib_set = _as_set(old_bib)

    print("=== Pickle comparison ===")
    print(f"NEW: {new_path}")
    print(f"OLD: {old_path}")
    print()

    print("--- Bibcodes (docs) ---")
    print(f"new docs: {len(new_bib):,} (unique: {len(new_bib_set):,})")
    print(f"old docs: {len(old_bib):,} (unique: {len(old_bib_set):,})")
    print(f"overlap docs (unique): {len(new_bib_set & old_bib_set):,}")
    print(f"jaccard docs: {_jaccard(new_bib_set, old_bib_set):.4f}")
    print(f"only in NEW (unique): {len(new_bib_set - old_bib_set):,}")
    print(f"only in OLD (unique): {len(old_bib_set - new_bib_set):,}")
    print()

    print("--- References (flattened) ---")
    new_ref_flat = _flatten_refs(new_refs)
    old_ref_flat = _flatten_refs(old_refs)
    new_ref_set = set(new_ref_flat)
    old_ref_set = set(old_ref_flat)
    print(f"new refs: {len(new_ref_flat):,} (unique: {len(new_ref_set):,})")
    print(f"old refs: {len(old_ref_flat):,} (unique: {len(old_ref_set):,})")
    print(f"overlap refs (unique): {len(new_ref_set & old_ref_set):,}")
    print(f"jaccard refs: {_jaccard(new_ref_set, old_ref_set):.4f}")
    print(f"only in NEW refs (unique): {len(new_ref_set - old_ref_set):,}")
    print(f"only in OLD refs (unique): {len(old_ref_set - new_ref_set):,}")
    print()

    print("--- Basic sanity checks ---")
    print(f"len(docs)==len(ref_lists): new {len(new_bib)==len(new_refs)} | old {len(old_bib)==len(old_refs)}")
    if new_esrc and old_esrc:
        print(f"len(docs)==len(esources_lists): new {len(new_bib)==len(new_esrc)} | old {len(old_bib)==len(old_esrc)}")
    if new_ft and old_ft:
        print(f"len(docs)==len(fulltext_urls): new {len(new_bib)==len(new_ft)} | old {len(old_bib)==len(old_ft)}")
    print()

    # Optional: show a few example bibcodes that differ (helps debugging)
    show_n = 20
    only_new = sorted(new_bib_set - old_bib_set)[:show_n]
    only_old = sorted(old_bib_set - new_bib_set)[:show_n]
    print(f"--- Example only-in-NEW bibcodes (first {show_n}) ---")
    print(only_new)
    print()
    print(f"--- Example only-in-OLD bibcodes (first {show_n}) ---")
    print(only_old)
    print()

    # Optional: duplicates (sometimes indicates paging issues)
    new_dups = [k for k, v in Counter(new_bib).items() if v > 1]
    old_dups = [k for k, v in Counter(old_bib).items() if v > 1]
    print("--- Duplicate bibcodes (should usually be empty) ---")
    print(f"new duplicates: {len(new_dups)}")
    print(f"old duplicates: {len(old_dups)}")

Only one run found: 2_data\bibcodes_references_20260120_124208.pkl
Create a second run (FORCE_REFRESH=True) to compare.
